# 11 · Filter by metadata
Level L2 · Part 2 · builds on lesson 10 (and the NumPy mask of lesson 06)

## Goal
By the end of this lesson you can answer "vegetarian recipes under 30 minutes, similar to *a
hearty
dinner*" with a Qdrant filter, and list every recipe that passes a filter with `scroll`.

## The idea
A **filter** is a set of yes/no rules on the payload: `vegetarian` is true, `minutes` is at most
30.
Only points that pass the rules can be results. Where the filter runs matters. **Post-filtering**
searches first and throws away the failures afterwards, so pages come back short or empty.
**Pre-filtering** decides who may compete before ranking, so the page fills up whenever enough
points qualify. It is the shop website you know: tick "vegetarian" and "under 30 min" in the
sidebar first, then sort the remaining dishes by relevance. A **payload index** is the sidebar's
lookup table that makes those ticks fast on a real server.

## Picture

```mermaid
flowchart LR
    subgraph post["Post-filtering (Step 2)"]
        A1["rank all 50"] --> A2["keep top 5"]
        A2 --> A3["drop non-matching"] --> A4["0 to 5 results"]
    end
    subgraph pre["Filter in the query (Step 3)"]
        B1["filter: vegetarian AND minutes <= 30<br/>(25 of 50 pass)"]
        B1 --> B2["rank only those 25"] --> B3["top 5: a full page"]
    end
```

Same query, same data, same k. Only the moment the rules are applied differs.

## Step by step
### Step 1 · The `recipes` collection, with descriptions
Rebuild lesson 10's collection in this lesson's own folder. This time the payload also carries the
recipe's `description`, so Step 7 can search inside it.

In [1]:
import shutil, warnings
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore", message="IProgress")     # quiet a progress-bar notice on import
from qdrant_client import QdrantClient, models

recipes = pd.read_csv("../data/recipes_50.csv")
saved = np.load("../data/recipe_embeddings_minilm.npz")
doc_emb, MODEL = saved["doc_emb"], str(saved["model"])
cache = np.load("../data/query_embeddings_minilm.npz")
query_cache = dict(zip(cache["queries"].tolist(), cache["query_emb"]))
print("model:", MODEL, "| recipes:", doc_emb.shape, "| cached queries:", len(query_cache))

model: sentence-transformers/all-MiniLM-L6-v2 | recipes: (50, 384) | cached queries: 15


`to_payload` is lesson 09's helper plus `description`. Every value is a plain Python type.

In [2]:
DB_PATH = "../data/cache/qdrant_11"
shutil.rmtree(DB_PATH, ignore_errors=True)           # start from an empty folder every run
client = QdrantClient(path=DB_PATH)
client.create_collection("recipes", vectors_config=models.VectorParams(
    size=doc_emb.shape[1], distance=models.Distance.COSINE))

def to_payload(row):
    return {"title": str(row.title), "description": str(row.description),
            "cuisine": str(row.cuisine), "vegetarian": bool(row.vegetarian),
            "minutes": int(row.minutes), "embed_model": MODEL}

client.upsert("recipes", points=[models.PointStruct(id=int(row.id), vector=doc_emb[i].tolist(),
              payload=to_payload(row)) for i, row in enumerate(recipes.itertuples())])
print("points:", client.count("recipes").count)
print(client.retrieve("recipes", ids=[10])[0].payload)

points: 50
{'title': 'Red lentil dal', 'description': 'Red lentils simmered with turmeric, cumin and garlic into a warming golden stew.', 'cuisine': 'Indian', 'vegetarian': True, 'minutes': 30, 'embed_model': 'sentence-transformers/all-MiniLM-L6-v2'}


Our query is "a hearty dinner", which is not among the 15 cached queries. The next cell embeds it
with the same model and saves it (needs the model download; the saved file ships in `data/`, so
the
cell after it works without the download).

In [3]:
import os
os.environ.update(TQDM_DISABLE="1", HF_HUB_VERBOSITY="error")
from sentence_transformers import SentenceTransformer

new_queries = ["a hearty dinner"]
model = SentenceTransformer(MODEL)
np.savez("../data/query_embeddings_minilm_hearty.npz", model=MODEL,
         queries=np.array(new_queries), query_emb=model.encode_query(new_queries))
print("saved", len(new_queries), "query vector(s)")

⏭  Skipped: this cell needs [download]. Run it yourself, or re-run with TUT_ENABLE=download.


Load it into the same `query_cache`, after checking it came from the same model as the recipes:

In [4]:
extra = np.load("../data/query_embeddings_minilm_hearty.npz")
assert str(extra["model"]) == MODEL
query_cache.update(zip(extra["queries"].tolist(), extra["query_emb"]))
hearty = query_cache["a hearty dinner"].tolist()
print("cached queries now:", len(query_cache), "| vector length:", len(hearty))

cached queries now: 16 | vector length: 384


### Step 2 · Post-filtering: search, then drop
First the way lesson 06 warned about: take the top 5, then keep only vegetarian recipes of 30
minutes or less. `wanted` is the rule written in plain Python.

In [5]:
def wanted(payload):
    return payload["vegetarian"] and payload["minutes"] <= 30

top5 = client.query_points("recipes", query=hearty, limit=5).points
for h in top5:
    p = h.payload
    row = (f"{p['title']:<24} veg={p['vegetarian']!s:<5} {p['minutes']:>3} min"
           f"  score {h.score:.3f}  pass={wanted(p)}")
    print(row)
print("post-filtered page:", [h.payload["title"] for h in top5 if wanted(h.payload)])

Lasagne                  veg=False  90 min  score 0.446  pass=False
Beef stew                veg=False 180 min  score 0.435  pass=False
Minestrone               veg=True   50 min  score 0.423  pass=False
Full English breakfast   veg=False  30 min  score 0.407  pass=False
Chicken noodle soup      veg=False  40 min  score 0.405  pass=False
post-filtered page: []


The top 5 are Lasagne, Beef stew, Minestrone (vegetarian, but 50 minutes), Full English breakfast
and Chicken noodle soup. None passes, so the page is **empty**, even though half the collection
would qualify. That is **post-filtering**: the rules ran after the ranking had already picked
its 5.

### Step 3 · A filter inside the query
Lesson 07 gave a preview of this filter; now we take it apart. Write the same rules as a
`models.Filter` and hand it to `query_points` as `query_filter`. Each rule is a `FieldCondition`
on one payload key: `MatchValue` for an exact value, `Range` for numbers.

In [6]:
quick_veg = models.Filter(must=[
    models.FieldCondition(key="vegetarian", match=models.MatchValue(value=True)),
    models.FieldCondition(key="minutes", range=models.Range(lte=30)),
])
hits = client.query_points("recipes", query=hearty, query_filter=quick_veg, limit=5).points
for h in hits:
    print(f"{h.payload['title']:<24} {h.payload['minutes']:>3} min  score {h.score:.3f}")

Guacamole                 10 min  score 0.367
Chocolate chip cookies    25 min  score 0.354
Caesar salad              15 min  score 0.350
Red lentil dal            30 min  score 0.343
Margherita pizza          25 min  score 0.342


A full page of 5, every one vegetarian and quick: Guacamole, Chocolate chip cookies, Caesar salad,
Red lentil dal, Margherita pizza. The docs put it plainly: "the search is performed only among
those points that satisfy the filter condition". That is **pre-filtering** in effect. (How an
approximate index keeps this fast and correct is lesson 34.) Note the scores: 0.342 to 0.367,
lower
than the unfiltered 0.446. The filter can only remove good matches, never add better ones.

`count` takes the same filter and tells you how many points it lets through:

In [7]:
n_pass = client.count("recipes", count_filter=quick_veg).count
mask = recipes["vegetarian"] & (recipes["minutes"] <= 30)          # lesson 06's NumPy mask
print("Qdrant count:", n_pass, "| NumPy mask:", int(mask.sum()))

Qdrant count: 25 | NumPy mask: 25


25 recipes pass, the same 25 as lesson 06's mask. Every condition in `must` has to hold: `must` is
AND.

### Step 4 · `should`: at least one must hold
`should` is OR. Ask for vegetarian recipes that are Indian **or** Italian. `MatchAny` says the
same
thing in one condition (SQL's `IN`); both should give the same list.

In [8]:
veg = models.FieldCondition(key="vegetarian", match=models.MatchValue(value=True))
with_should = models.Filter(must=[veg], should=[
    models.FieldCondition(key="cuisine", match=models.MatchValue(value="Indian")),
    models.FieldCondition(key="cuisine", match=models.MatchValue(value="Italian"))])
with_any = models.Filter(must=[
    veg,
    models.FieldCondition(key="cuisine", match=models.MatchAny(any=["Indian", "Italian"]))])

def top_titles(flt, k=5):
    hits = client.query_points("recipes", query=hearty, query_filter=flt, limit=k).points
    return [h.payload["title"] for h in hits]

a, b = top_titles(with_should), top_titles(with_any)
print("should  :", a)
print("MatchAny:", b, "| same:", a == b)

should  : ['Minestrone', 'Tomato soup', 'Caesar salad', 'Red lentil dal', 'Margherita pizza']
MatchAny: ['Minestrone', 'Tomato soup', 'Caesar salad', 'Red lentil dal', 'Margherita pizza'] | same: True


Minestrone and Tomato soup are back (no time limit now). Inside one `Filter`, the three clauses
are
combined with AND: every `must`, at least one `should`, and no `must_not`.

### Step 5 · `must_not`: exclude
Chocolate chip cookies is not much of a dinner. Keep Step 3's rules and exclude American recipes
with `must_not`:

In [9]:
quick_veg_not_us = models.Filter(must=quick_veg.must, must_not=[
    models.FieldCondition(key="cuisine", match=models.MatchValue(value="American"))])
hits = client.query_points("recipes", query=hearty, query_filter=quick_veg_not_us, limit=5).points
print([(h.payload["title"], h.payload["cuisine"]) for h in hits])
print("passing points:", client.count("recipes", count_filter=quick_veg_not_us).count)

[('Guacamole', 'Mexican'), ('Caesar salad', 'Italian'), ('Red lentil dal', 'Indian'), ('Margherita pizza', 'Italian'), ('Mango salsa', 'Mexican')]
passing points: 20


The cookies are gone and Mango salsa moves up to fill the page; 20 recipes pass instead of 25.

### Step 6 · Ranges
`Range` takes `gt`, `gte`, `lt` and `lte` (greater than, greater or equal, less than, less or
equal), in any combination, on integer or float fields. Count a few time windows:

In [10]:
windows = {"under 15 (lt=15)": models.Range(lt=15),
           "15 to 30 (gte=15, lte=30)": models.Range(gte=15, lte=30),
           "over 30 and up to 60 (gt=30, lte=60)": models.Range(gt=30, lte=60),
           "over 60 (gt=60)": models.Range(gt=60)}
for name, rng in windows.items():
    f = models.Filter(must=[models.FieldCondition(key="minutes", range=rng)])
    print(f"{name:<38} {client.count('recipes', count_filter=f).count:>2} recipes")

under 15 (lt=15)                        8 recipes
15 to 30 (gte=15, lte=30)              24 recipes
over 30 and up to 60 (gt=30, lte=60)   10 recipes
over 60 (gt=60)                         8 recipes


The four windows do not overlap and cover everything: 8 + 24 + 10 + 8 = 50. Mixing up `lt` and
`lte` is the classic off-by-one: a 30-minute recipe is in `lte=30` but not in `lt=30`.

### Step 7 · Full-text match: words inside a description
`MatchValue` compares the whole value. To ask "does the description mention lentils?", use
`MatchText`. Without a text index Qdrant splits the text into lowercase words and requires every
query word to appear as a **whole word**:

In [11]:
def titles_where(field, match, k=10):
    f = models.Filter(must=[models.FieldCondition(key=field, match=match)])
    return [p.payload["title"] for p in client.scroll("recipes", scroll_filter=f, limit=k)[0]]

for text in ["lentils", "Lentils", "lentil", "coconut milk"]:
    print(f"MatchText({text!r:<15}) ->", titles_where("description", models.MatchText(text=text)))

MatchText('lentils'      ) -> ['Red lentil dal']
MatchText('Lentils'      ) -> ['Red lentil dal']
MatchText('lentil'       ) -> []
MatchText('coconut milk' ) -> ['Butternut squash soup', 'Thai green curry']


Case does not matter ("Lentils" finds Red lentil dal), but "lentil" finds nothing: the description
says "lentils", and there is no stemming. "coconut milk" requires both words somewhere. Two
relatives: `MatchTextAny` needs **any** of the words, and `MatchPhrase` needs the words **next to
each other, in order**. On a server, create a text index for these (Step 9) so all three behave as
shown here:

In [12]:
legumes = models.MatchTextAny(text_any="beans lentils chickpeas chickpea")
print("any legume :", titles_where("description", legumes))
print("phrase     :", titles_where("description", models.MatchPhrase(phrase="coconut milk")))
print("wrong order:", titles_where("description", models.MatchPhrase(phrase="milk coconut")))

any legume : ['Minestrone', 'Red lentil dal', 'Chili con carne', 'Quinoa salad', 'Full English breakfast', 'Falafel wrap', 'Hummus']
phrase     : ['Butternut squash soup', 'Thai green curry']
wrong order: []


A full-text condition is just another `FieldCondition`, so it combines with the rest. A hearty,
quick, vegetarian dinner with a legume in it:

In [13]:
filling = models.Filter(
    must=quick_veg.must + [models.FieldCondition(key="description", match=legumes)])
hits = client.query_points("recipes", query=hearty, query_filter=filling, limit=5).points
print([(h.payload["title"], h.payload["minutes"], round(h.score, 3)) for h in hits])

[('Red lentil dal', 30, 0.343), ('Falafel wrap', 30, 0.32), ('Quinoa salad', 20, 0.311), ('Hummus', 10, 0.308)]


Only 4 recipes pass all three rules, so `limit=5` returns 4: a filter can leave fewer points than
`limit`, and that is a correct answer, not a bug.

### Step 8 · Prefix match on a keyword
`MatchPrefix` (new in Qdrant 1.19) matches keyword values that **start with** a string. Per the
docs it is "byte-wise and case-sensitive", like `MatchValue`:

In [14]:
for prefix in ["Chocolate", "chocolate", "Ma"]:
    found = titles_where("title", models.MatchPrefix(prefix=prefix))
    print(f"title starts with {prefix!r:<12} ->", found)

title starts with 'Chocolate'  -> ['Chocolate brownies', 'Chocolate chip cookies']
title starts with 'chocolate'  -> []
title starts with 'Ma'         -> ['Mango salsa', 'Mango lassi', 'Margherita pizza']


Two chocolate recipes for "Chocolate", none for "chocolate". Use prefixes for codes and ids
("SKU-12…", "2026-10-…"), or for type-ahead on clean values. For words *inside* text, use Step 7.

### Step 9 · Payload indexes for the fields you filter on
On a Qdrant **server** each filtered field should get a **payload index**: a lookup table from
value
to points, so a filter does not read every payload. You pick a type per field. Ours:

In [15]:
index_plan = {
    "vegetarian": models.PayloadSchemaType.BOOL,
    "minutes": models.IntegerIndexParams(type=models.IntegerIndexType.INTEGER,
                                         lookup=True, range=True),
    "cuisine": models.PayloadSchemaType.KEYWORD,
    "title": models.KeywordIndexParams(type=models.KeywordIndexType.KEYWORD, prefix=True),
    "description": models.TextIndexParams(type=models.TextIndexType.TEXT,
                                          tokenizer=models.TokenizerType.WORD, lowercase=True,
                                          phrase_matching=True),
}
for field, schema in index_plan.items():
    if isinstance(schema, models.PayloadSchemaType):
        shown = schema.value
    else:
        shown = schema.model_dump(exclude_none=True, mode="json")
    print(f"{field:<12} -> {shown}")

vegetarian   -> bool
minutes      -> {'type': 'integer', 'lookup': True, 'range': True}
cuisine      -> keyword
title        -> {'type': 'keyword', 'prefix': True}
description  -> {'type': 'text', 'tokenizer': 'word', 'lowercase': True, 'phrase_matching': True}


`minutes` needs `range=True` for `Range`; `title` needs `prefix=True` for Step 8; `description`
gets a full-text index; `phrase_matching=True` lets the server answer `MatchPhrase` (Step 7). Now
create them. In local mode, watch the warning:

In [16]:
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    for field, schema in index_plan.items():
        result = client.create_payload_index("recipes", field_name=field, field_schema=schema)
print("warning:", str(caught[0].message))
print("last result:", result)
print("payload_schema in local mode:", client.get_collection("recipes").payload_schema)

last result: operation_id=0 status=<UpdateStatus.COMPLETED: 'completed'>
payload_schema in local mode: {}


The call succeeds, but local mode warns "Payload indexes have no effect in the local Qdrant" (the
warning text is printed above) and `payload_schema` stays `{}`: local mode scans every payload. On
a server, the same code builds the indexes and `get_collection` lists them. The docs advise
creating them "before ingesting data", because the server builds its search index with them in
mind (lesson 34 shows why this matters), and indexing only the fields you filter on, since each
one costs memory. Here we add them after loading only to show the call; in real code call
`create_payload_index` right after `create_collection`, before the first upsert.

A server can also be set to a **strict mode** that rejects filters on fields without an index.
Lesson 34 shows why the index matters for an approximate search.

### Step 10 · `scroll`: page through every match
`query_points` ranks by similarity. Sometimes you want *all* points that pass a filter, with no
query vector: an export, a "browse vegetarian" page. That is `scroll`. It returns a pair:

In [17]:
veg_only = models.Filter(must=[veg])
records, next_offset = client.scroll("recipes", scroll_filter=veg_only, limit=10)
print("page 1 ids:", [r.id for r in records])
print("next_page_offset:", next_offset)
first = records[0]
print("first record:", first.id, first.payload["title"], "| has score:", hasattr(first, "score"))

page 1 ids: [1, 2, 3, 4, 5, 6, 10, 11, 13, 14]
next_page_offset: 15
first record: 1 Tomato soup | has score: False


A list of `Record`s, ordered by id. `has score: False`: a `Record` has no score, because nothing
was ranked. The second item is the **next page offset**: the id to start the next page from. Pass
it back as `offset` and repeat until it is `None`:

In [18]:
pages, offset = [], None
while True:
    records, offset = client.scroll("recipes", scroll_filter=veg_only, limit=10, offset=offset)
    pages.append([r.id for r in records])
    if offset is None:
        break
for i, ids in enumerate(pages, 1):
    print(f"page {i}: {len(ids):>2} ids {ids}")
n_veg = client.count("recipes", count_filter=veg_only).count
print("total:", sum(map(len, pages)), "| count says:", n_veg)

page 1: 10 ids [1, 2, 3, 4, 5, 6, 10, 11, 13, 14]
page 2: 10 ids [15, 16, 17, 18, 19, 20, 21, 22, 23, 27]
page 3: 10 ids [28, 29, 30, 31, 32, 35, 36, 38, 44, 45]
page 4:  3 ids [46, 47, 50]
total: 33 | count says: 33


Four pages: 10 + 10 + 10 + 3 = 33 vegetarian recipes, matching `count`. The last page is short and
its offset is `None`, the stop signal.

Because the offset is an id, not a page number, the next page always starts at that id. See what
that means when points arrive mid-export: fetch page 1, then upsert two vegetarian points, id 0
(below the offset) and id 99 (above it), then finish the loop.

In [19]:
records, offset = client.scroll("recipes", scroll_filter=veg_only, limit=10)
seen = [r.id for r in records]
print("after page 1, offset =", offset)
client.upsert("recipes", points=[
    models.PointStruct(id=i, vector=doc_emb[0].tolist(), payload={"title": f"New dish {i}",
                       "vegetarian": True, "minutes": 20, "cuisine": "Test"}) for i in (0, 99)])
while offset is not None:
    records, offset = client.scroll("recipes", scroll_filter=veg_only, limit=10, offset=offset)
    seen += [r.id for r in records]
print("ids seen:", len(seen), "| 0 seen:", 0 in seen, "| 99 seen:", 99 in seen)
client.delete("recipes", points_selector=models.PointIdsList(points=[0, 99]))
print("after cleanup, vegetarian count:", client.count("recipes", count_filter=veg_only).count)

after page 1, offset = 15
ids seen: 34 | 0 seen: False | 99 seen: True
after cleanup, vegetarian count: 33


Id 0 was never visited in this pass, because paging had already moved past it; id 99 appears on
the last page. 34 ids = the 33 recipes + id 99, so no point was repeated. We deleted both so the
rest of the lesson sees the same 50 recipes.

### Step 11 · Paging a similarity search
`scroll` pages by id. For **similarity** results, `query_points` has its own `offset`, which skips
the first results. Page 2 of the hearty search:

In [20]:
page1 = client.query_points("recipes", query=hearty, query_filter=quick_veg, limit=5).points
page2 = client.query_points("recipes", query=hearty, query_filter=quick_veg,
                            limit=5, offset=5).points
print("page 1:", [h.payload["title"] for h in page1])
print("page 2:", [h.payload["title"] for h in page2])
print("scores keep falling:", round(page1[-1].score, 3), ">=", round(page2[0].score, 3))

page 1: ['Guacamole', 'Chocolate chip cookies', 'Caesar salad', 'Red lentil dal', 'Margherita pizza']
page 2: ['Mango salsa', 'Mango lassi', 'Gazpacho', 'Falafel wrap', 'Greek salad']
scores keep falling: 0.342 >= 0.339


Page 2 continues where page 1 stopped. The docs warn that "large offset values may cause
performance
issues": the server still finds the first `offset + limit` results. Keep search paging shallow.

## What just happened
- Step 2: post-filtering the top 5 left an **empty** page.
- Step 3: `query_filter` returned a full page; `count` gave 25, like lesson 06's mask.
- Steps 4–6: `should` matched `MatchAny`; `must_not` cut 25 to 20; four ranges split 50 into
  8 + 24 + 10 + 8.
- Steps 7–8: `MatchText` ignored case but needed whole words; `MatchPrefix` was case-sensitive.
- Step 9: `create_payload_index` only warned "no effect" in local mode.
- Steps 10–11: `scroll` returned 33 vegetarian recipes in pages of 10 + 10 + 10 + 3; a point
  inserted below the offset was skipped; `query_points(offset=5)` gave search page 2.

## Common mistakes

**1. Wrong case in a keyword match.** `MatchValue` on a string is exact, byte for byte. The
payload says `"Italian"`. Compare the wrong case with the right one:

In [21]:
def cuisine_is(name):
    return models.Filter(must=[models.FieldCondition(key="cuisine",
                                                     match=models.MatchValue(value=name))])

print("'italian' hits:", top_titles(cuisine_is("italian")))
print("'Italian' hits:", top_titles(cuisine_is("Italian")))

'italian' hits: []
'Italian' hits: ['Lasagne', 'Minestrone', 'Tomato soup', 'Spaghetti carbonara', 'Caesar salad']


The lowercase filter gives an empty list. Qdrant raises no error, so this bug is silent: the page
is just empty. Fix: store keyword values in one agreed case (and normalise user input to it), or
use full-text matching.

**2. Writing the range on the condition itself.** `lte=30` belongs inside a `Range`, not on the
`FieldCondition`:

In [22]:
try:
    quick = models.FieldCondition(key="minutes", lte=30)
except Exception as err:  # this mistake is on purpose: show the error, keep the notebook running
    print(f"{type(err).__name__}: {err}")

ValidationError: 1 validation error for FieldCondition
lte
  Extra inputs are not permitted [type=extra_forbidden, input_value=30, input_type=int]
    For further information visit https://errors.pydantic.dev/2.13/v/extra_forbidden


It prints a pydantic `ValidationError` ending in `lte  Extra inputs are not permitted`. Fix:
`range=models.Range(lte=30)`, as in Step 3.

In [23]:
quick = models.FieldCondition(key="minutes", range=models.Range(lte=30))
n_quick = client.count("recipes", count_filter=models.Filter(must=[quick])).count
print(n_quick, "recipes take 30 minutes or less")

32 recipes take 30 minutes or less


**3. Looping over `scroll` as if it were a list of points.** It returns a *pair*: the records and
the
next offset.

In [24]:
try:
    for record in client.scroll("recipes", scroll_filter=veg_only, limit=3):
        print(record.payload["title"])
except Exception as err:  # this mistake is on purpose: show the error, keep the notebook running
    print(f"{type(err).__name__}: {err}")

AttributeError: 'list' object has no attribute 'payload'


It prints `AttributeError: 'list' object has no attribute 'payload'`: the first item of the pair
is
the whole list. Fix: unpack it.

In [25]:
records, next_offset = client.scroll("recipes", scroll_filter=veg_only, limit=3)
for record in records:
    print(record.payload["title"])
print("next page starts at id", next_offset)

Tomato soup
Mango salsa
Chocolate brownies
next page starts at id 4


## Try it
1. **A warm weekday dish.** For the cached query "something warm for a cold day", return up to 5
   vegetarian recipes that take **from 30 to 60 minutes** (both included) and are **not French**.
   Print title, minutes and score.
2. **Export the chili recipes.** With `scroll` and pages of 3, collect every recipe whose
   description mentions "chili" (full-text). Print each page, then check the total against `count`
   with the same filter.

In [26]:
# your code here

Close the client so the local folder is released.

In [27]:
client.close()
print("closed")

closed


## Key terms
- **filter (Qdrant)** — `models.Filter`: payload rules a point must pass; used as `query_filter`,
  `count_filter` or `scroll_filter`.
- **FieldCondition** — one rule on one payload key: a `match` or a `range`.
- **must / should / must_not** — a filter's clauses: AND, OR and NOT.
- **range condition** — `models.Range(gt, gte, lt, lte)`: numeric bounds on a field.
- **MatchAny** — exact match against a list of values (SQL `IN`).
- **full-text match** — `MatchText` / `MatchTextAny` / `MatchPhrase`: words inside a text field.
- **prefix match** — `MatchPrefix`: a keyword value starts with a string; case-sensitive.
- **post-filtering** — search first, then remove failures; pages come back short.
- **pre-filtering** — filter before or during ranking, so only passing points compete.
- **payload index** — a per-field lookup that speeds up filters on a server; no effect in local
  mode.
- **scroll** — read all points matching a filter, page by page, with no query vector.
- **next page offset** — the id where `scroll`'s next page starts; `None` on the last page.